<a href="https://colab.research.google.com/github/secrabdou/GPT/blob/main/GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/build-nanogpt/master/hellaswag.py

--2026-09-22 18:08:11--  https://raw.githubusercontent.com/karpathy/build-nanogpt/master/hellaswag.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7713 (7.5K) [text/plain]
Saving to: ‘hellaswag.py.3’

hellaswag.py.3      100%[===================>]   7.53K  --.-KB/s    in 0s      

2026-09-22 18:08:11 (97.4 MB/s) - ‘hellaswag.py.3’ saved [7713/7713]



In [ ]:


import os
import numpy as np

os.makedirs("edu_fineweb10B", exist_ok=True)
# Generate random tokens representing vocabulary indices
dummy_tokens = np.random.randint(0, 50257, size=(1000000,), dtype=np.int32)

np.save("edu_fineweb10B/edufineweb_train_000001.npy", dummy_tokens)
np.save("edu_fineweb10B/edufineweb_val_000001.npy", dummy_tokens)
print("Dummy dataset created successfully!")

Dummy dataset created successfully!


In [7]:
from dataclasses import dataclass
import inspect
import math
import os
import time
import numpy as np

import torch
import torch.distributed as dist
import torch.nn as nn
from torch.distributed import destroy_process_group, init_process_group
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from hellaswag import render_example, iterate_examples

import tiktoken

enc = tiktoken.get_encoding("gpt2")


# --- Model Architecture ---

class CasualSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.block_size, config.block_size)).view(
                1, 1, config.block_size, config.block_size
            ),
        )

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True) # Corrected function name
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.proj(x)
        return x


class block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.att = CasualSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 65
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(
            dict(
                wte=nn.Embedding(config.vocab_size, config.n_embd),
                wpe=nn.Embedding(config.block_size, config.n_embd),
                h=nn.ModuleList([block(config) for _ in range(config.n_layer)]),
                ln_f=nn.LayerNorm(config.n_embd),
            )
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, "NANOGPT_SCALE_INIT"):
                std *= (2 * self.config.n_layer) ** -0.5
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}"

        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        wte = self.transformer.wte(idx)
        wpe = self.transformer.wpe(pos)
        x = wte + wpe
        for block_layer in self.transformer.h:
            x = block_layer(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
        return logits

    def configure_optimizers(self, weight_decay, learning_rate, device):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}

        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": nodecay_params, "weight_decay": 0.0},
        ]

        fused_available = "fused" in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device.startswith("cuda")
        if master_process:
            print(f"using fused AdamW: {use_fused}")

        optimizer_kwargs = {"lr": learning_rate, "betas": (0.9, 0.95), "eps": 1e-8}
        if use_fused:
            optimizer_kwargs["fused"] = True

        optimizer = torch.optim.AdamW(optim_groups, **optimizer_kwargs)
        return optimizer


# --- Data Loading Utilities ---

def load_tokens(filename):
    npt = np.load(filename)
    ptt = torch.tensor(npt, dtype=torch.long)
    return ptt


class Dataloaderlite:
    def __init__(self, B, T, process_rank, num_processes, split):
        self.B = B
        self.T = T
        self.process_rank = process_rank
        self.num_processes = num_processes
        assert split in {'train', 'val'}
        data_root = "edu_fineweb10B"
        shards = os.listdir(data_root)  # Fixed typo: os.listdir
        shards = [s for s in shards if split in s]
        shards = sorted(shards)
        shards = [os.path.join(data_root, s) for s in shards]
        self.shards = shards
        assert len(shards) > 0, f"no shards found for split {split}"
        if master_process:
            print(f"found {len(shards)} shards for split {split}")
        self.reset()

    def reset(self):
        self.current_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.current_position = self.B * self.T * self.process_rank

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position : self.current_position + B * T + 1]
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)

        # Advance sequence position by global stride across all GPUs
        self.current_position += B * T * self.num_processes

        # Advance shard if out of bounds
        if self.current_position + (B * T * self.num_processes + 1) > len(self.tokens):
            self.current_shard = (self.current_shard + 1) % len(self.shards)
            self.tokens = load_tokens(self.shards[self.current_shard])
            self.current_position = B * T * self.process_rank

        return x, y


# --- HellaSwag Helper ---

def get_most_likely_row(tokens, mask, logits):
    # Evaluate autoregressive loss at all sequence positions
    shift_logits = (logits[..., :-1, :]).contiguous()
    shift_tokens = (tokens[..., 1:]).contiguous()
    flat_shift_logits = shift_logits.view(-1, shift_logits.size(-1))
    flat_shift_tokens = shift_tokens.view(-1)
    shift_losses = F.cross_entropy(flat_shift_logits, flat_shift_tokens, reduction='none')
    shift_losses = shift_losses.view(tokens.size(0), -1)

    # Apply completion mask
    shift_mask = (mask[..., 1:]).contiguous()
    masked_shift_losses = shift_losses * shift_mask
    sum_loss = masked_shift_losses.sum(dim=1)
    avg_loss = sum_loss / shift_mask.sum(dim=1)
    pred_norm = avg_loss.argmin().item()
    return pred_norm


# --- Environment & DDP Setup ---

ddp = "RANK" in os.environ and "WORLD_SIZE" in os.environ
if ddp:
    assert torch.cuda.is_available(), "DDP requires CUDA"
    init_process_group(backend="nccl")
    ddp_rank = int(os.environ["RANK"])
    ddp_local_rank = int(os.environ["LOCAL_RANK"])
    ddp_world_size = int(os.environ["WORLD_SIZE"])
    device = f"cuda:{ddp_local_rank}"
    torch.cuda.set_device(device)
    master_process = ddp_rank == 0
else:
    ddp_rank = 0
    ddp_local_rank = 0
    ddp_world_size = 1
    master_process = True
    device = "cpu"
    if torch.cuda.is_available():
        device = "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"

device_type = "cuda" if device.startswith("cuda") else "cpu"

# Set up logging directory
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, "log.txt")

torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

total_batch_size = 524288
B = 8 # Reduced from 16 to 8 to mitigate OutOfMemoryError
T = 512 # Reduced from 1024 to 512 to further mitigate OutOfMemoryError
assert total_batch_size % (B * T * ddp_world_size) == 0, "total_batch_size must be divisible by B * T * ddp_world_size"
grad_accum_steps = total_batch_size // (B * T * ddp_world_size)

if master_process:
    print(f"total desired batch size : {total_batch_size}")
    print(f"=> calculated gradient accumulation steps : {grad_accum_steps}")

# Corrected: Instantiate train and val loaders independently
train_loader = Dataloaderlite(B=B, T=T, process_rank=ddp_rank, num_processes=ddp_world_size, split='train')
val_loader = Dataloaderlite(B=B, T=T, process_rank=ddp_rank, num_processes=ddp_world_size, split='val')

torch.set_float32_matmul_precision("high")

# --- Model & Optimizer Initialization ---

model = GPT(GPTConfig(vocab_size=50304, block_size=T))
model.to(device)

use_compile = True
if use_compile:
    model = torch.compile(model)

if ddp:
    model = DDP(model, device_ids=[ddp_local_rank])

# Convenient uncompiled alias
raw_model = model.module if ddp else (model._orig_mod if hasattr(model, '_orig_mod') else model)

optimizer = raw_model.configure_optimizers(weight_decay=0.1, learning_rate=6e-4, device=device)

# --- Hyperparameters & LR Schedule ---

max_lr = 6e-4 * 3
min_lr = max_lr * 0.1
warmup_steps = 10
max_steps = 50

def get_lr(it):
    if it < warmup_steps:
        return max_lr * (it + 1) / warmup_steps
    if it > max_steps:
        return min_lr
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)


# --- Training & Evaluation Loop ---

for step in range(max_steps):
    t0 = time.time()
    last_step = (step == max_steps - 1)

    # ==========================================
    # 1. VALIDATION LOOP
    # ==========================================
    if step % 250 == 0 or last_step:
        model.eval()
        val_loader.reset()
        val_loss_accum = torch.tensor(0.0, device=device)
        val_loss_steps = 20

        with torch.no_grad():
            for _ in range(val_loss_steps):
                x, y = val_loader.next_batch()
                x, y = x.to(device), y.to(device)

                with torch.autocast(device_type=device_type, dtype=torch.bfloat16, enabled=(device_type == "cuda")):
                    logits, loss = model(x, y)

                loss = loss / val_loss_steps
                val_loss_accum += loss.detach()

        if ddp:
            dist.all_reduce(val_loss_accum, op=dist.ReduceOp.AVG)

        if master_process:
            print(f"validation loss: {val_loss_accum.item():.4f}")
            with open(log_file, "a") as f:
                f.write(f"{step} val {val_loss_accum.item():.4f}\n")
            if step > 0 and (step % 5000 == 0 or last_step):
                checkpoint_path = os.path.join(log_dir, f"model_{step:05d}.pt")
                checkpoint = {
                    'model': raw_model.state_dict(),
                    'config': raw_model.config,
                    'step': step,
                    'val_loss': val_loss_accum.item()
                }
                torch.save(checkpoint, checkpoint_path)

    # ==========================================
    # 2. HELLASWAG EVALUATION
    # ==========================================
    if (step > 0 and step % 250 == 0) or last_step:
        model.eval()
        num_correct_norm = 0
        num_total = 0

        for i, example in enumerate(iterate_examples("val")):
            if i % ddp_world_size != ddp_rank:
                continue

            _, tokens, mask, label = render_example(example)
            tokens = tokens.to(device)
            mask = mask.to(device)

            with torch.no_grad():
                with torch.autocast(device_type=device_type, dtype=torch.bfloat16, enabled=(device_type == "cuda")):
                    logits = raw_model(tokens)

                pred_norm = get_most_likely_row(tokens, mask, logits)

            num_total += 1
            num_correct_norm += int(pred_norm == label)

        if ddp:
            num_total = torch.tensor(num_total, dtype=torch.long, device=device)
            num_correct_norm = torch.tensor(num_correct_norm, dtype=torch.long, device=device)
            dist.all_reduce(num_total, op=dist.ReduceOp.SUM)
            dist.all_reduce(num_correct_norm, op=dist.ReduceOp.SUM)
            num_total = num_total.item()
            num_correct_norm = num_correct_norm.item()

        acc_norm = num_correct_norm / num_total if num_total > 0 else 0.0

        if master_process:
            print(f"HellaSwag accuracy: {num_correct_norm}/{num_total}={acc_norm:.4f}")
            with open(log_file, "a") as f:
                f.write(f"{step} hella {acc_norm:.4f}\n")

    # ==========================================
    # 3. SAMPLE GENERATION
    # ==========================================
    if (step > 0 and step % 250 == 0) or last_step:
        model.eval()
        num_return_sequences = 4
        max_length = 32
        tokens = enc.encode("Hello, I'm a language model,")
        tokens = torch.tensor(tokens, dtype=torch.long)
        tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1)
        xgen = tokens.to(device)

        sample_rng = torch.Generator(device=device)
        sample_rng.manual_seed(42 + ddp_rank)

        while xgen.size(1) < max_length:
            with torch.no_grad():
                with torch.autocast(device_type=device_type, dtype=torch.bfloat16, enabled=(device_type == "cuda")):
                    # Corrected: Call raw_model without targets returns logits tensor directly
                    logits = raw_model(xgen)

                logits = logits[:, -1, :]
                probs = F.softmax(logits, dim=-1)
                topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
                ix = torch.multinomial(topk_probs, 1, generator=sample_rng)
                xcol = torch.gather(topk_indices, -1, ix)
                xgen = torch.cat((xgen, xcol), dim=1)

        if master_process:
            for i in range(num_return_sequences):
                gen_tokens = xgen[i, :max_length].tolist()
                decoded = enc.decode(gen_tokens)
                print(f"sample {i}: {decoded}")

    # ==========================================
    # 4. TRAINING STEP
    # ==========================================
    model.train()
    optimizer.zero_grad()
    loss_accum = torch.tensor(0.0, device=device)

    for micro_step in range(grad_accum_steps):
        x, y = train_loader.next_batch()
        x, y = x.to(device), y.to(device)

        with torch.autocast(device_type=device_type, dtype=torch.bfloat16, enabled=(device_type == "cuda")):
            logits, loss = model(x, y)

        loss = loss / grad_accum_steps
        loss_accum += loss.detach()

        if ddp:
            model.require_backward_grad_sync = (micro_step == grad_accum_steps - 1)

        loss.backward()

    if ddp:
        dist.all_reduce(loss_accum, op=dist.ReduceOp.AVG)

    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # Learning rate schedule update
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    optimizer.step()

    if device.startswith("cuda"):
        torch.cuda.synchronize()

    t1 = time.time()
    dt = t1 - t0
    tokens_processed = train_loader.B * train_loader.T * grad_accum_steps * ddp_world_size
    tokens_per_sec = tokens_processed / dt

    if master_process:
        print(f"step {step:4d} | loss: {loss_accum.item():.6f} | lr: {lr:.4e} | norm: {norm:.4f} | dt: {dt * 1000:.2f}ms | tok/sec: {tokens_per_sec:.2f}")
        with open(log_file, "a") as f:
            f.write(f"{step} train {loss_accum.item():.6f}\n")

if ddp:
    destroy_process_group()

total desired batch size : 524288
=> calculated gradient accumulation steps : 128
found 1 shards for split train
found 1 shards for split val
using fused AdamW: True


/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:3403: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(


validation loss: 10.9031


/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:3403: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:3403: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/_inductor/compile_fx.py:3403: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(


step    0 | loss: 10.901829 | lr: 1.8000e-04 | norm: 0.1809 | dt: 62003.16ms | tok/sec: 8455.83
step    1 | loss: 10.898341 | lr: 3.6000e-04 | norm: 0.1374 | dt: 48206.10ms | tok/sec: 10875.97
step    2 | loss: 10.867255 | lr: 5.4000e-04 | norm: 0.1865 | dt: 48335.16ms | tok/sec: 10846.93
step    3 | loss: 10.864932 | lr: 7.2000e-04 | norm: 0.8048 | dt: 48147.72ms | tok/sec: 10889.15
step    4 | loss: 10.821041 | lr: 9.0000e-04 | norm: 0.2614 | dt: 48173.69ms | tok/sec: 10883.28
step    5 | loss: 10.826252 | lr: 1.0800e-03 | norm: 0.7632 | dt: 48372.06ms | tok/sec: 10838.65
step    6 | loss: 10.777929 | lr: 1.2600e-03 | norm: 0.4228 | dt: 48315.48ms | tok/sec: 10851.35
step    7 | loss: 10.788400 | lr: 1.4400e-03 | norm: 0.8498 | dt: 48414.78ms | tok/sec: 10829.09
step    8 | loss: 10.775116 | lr: 1.6200e-03 | norm: 0.7363 | dt: 48388.68ms | tok/sec: 10834.93
step    9 | loss: 10.781783 | lr: 1.8000e-03 | norm: 0.9811 | dt: 48333.71ms | tok/sec: 10847.25
step   10 | loss: 10.790219 | l

/content/hellaswag/hellaswag_val.jsonl: 100%|██████████| 14.0/14.0 [00:00<00:00, 37.8kiB/s]


JSONDecodeError: Extra data: line 1 column 4 (char 3)